# LLM Agents with LangChain and GeminiAI

In [ ]:
# Install the necessary pacakges
! pip install langchain -qU
! pip install langchain-google-genai -qU
! pip install langchain-community -qU

! pip install duckduckgo-search -qU # google search
! pip install pyowm -qU # weather

! pip install langchainhub -qU #to get a default prompt

In [ ]:
import os
from google.colab import userdata

# Initialize GeminiAI LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Set OpenAI API Key
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Initialize the GenAI model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

# Initialize DuckDuckGo Search Tool

In [ ]:
!pip install -U ddgs

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# Intialize DuckDuckGo search tool
search_tool = DuckDuckGoSearchRun()

/tmp/ipykernel_9925/3495245340.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
search_tool

DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'))

# Initialize OpenWeatherMap Tool

In [ ]:
from langchain_community.utilities import OpenWeatherMapAPIWrapper
from langchain_core.tools import StructuredTool

# Set API key
os.environ["OPENWEATHERMAP_API_KEY"] = userdata.get("OPENWEATHER")

# Initialize API wrapper
weather = OpenWeatherMapAPIWrapper(
    openweathermap_api_key=userdata.get("OPENWEATHER")
)

# Create tool properly
weather_tool = StructuredTool.from_function(
    func=weather.run,
    name="openweathermap",
    description="Get weather for a city"
)

# Load Prompt Template from LangChainHUb

In [ ]:
from google.colab import userdata
from langsmith import Client

# Initialize LangSmith client
client = Client(api_key=userdata.get("LANGSMITH_API_KEY"))

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)
print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


# Create Built-In Agent

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install -U langchain langchain-classic langsmith langchain-core langchain-community

from langchain_classic.agents import AgentExecutor, create_react_agent
from langsmith import Client
from google.colab import userdata

# Pull the prompt (as set up earlier)
client = Client(api_key=userdata.get("LANGSMITH_API_KEY"))
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

# llm, search_tool, weather_tool need to be defined above this cell
tools = [search_tool, weather_tool]

# Construct the ReAct agent
agent = create_react_agent(llm, tools, prompt)

# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)

Invoke the agent executor with an example question

In [ ]:
response = agent_executor.invoke({"input": "What is the weather in London ?"})



> Entering new AgentExecutor chain...
Action: openweathermap
Action Input: LondonIn London, the current weather is as follows:
Detailed status: scattered clouds
Wind speed: 0.45 m/s, direction: 119°
Humidity: 79%
Temperature: 
  - Current: 17.14°C
  - High: 17.79°C
  - Low: 15.83°C
  - Feels like: 16.97°C
Rain: {}
Heat index: None
Cloud cover: 30%I now know the final answer
Final Answer: The weather in London is scattered clouds with a current temperature of 17.14°C, a high of 17.79°C, and a low of 15.83°C. The wind speed is 0.45 m/s from 119°, and humidity is 79%. Cloud cover is 30%.

> Finished chain.


In [ ]:
response = agent_executor.invoke({"input": "What is Machine Learning ?"})




> Entering new AgentExecutor chain...
Thought: The user is asking for a definition of Machine Learning. This is a general knowledge question and does not require real-time data or specific tools. I can define it directly.
Final Answer: Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention. Instead of being explicitly programmed for every task, ML algorithms build a mathematical model based on sample data, known as "training data," in order to make predictions or decisions without being explicitly programmed to perform the task. Key aspects include algorithms that can learn from data, iterative processes to improve performance, and the ability to generalize from learned examples to new, unseen data.

> Finished chain.


In [ ]:
response = agent_executor.invoke({"input": "How many letters in the word  Machine Learning"})



> Entering new AgentExecutor chain...
Thought: The user is asking to count the number of letters in the phrase "Machine Learning". I need to count all alphabetic characters, ignoring the space.
Machine has 7 letters.
Learning has 8 letters.
Total letters = 7 + 8 = 15.
I do not need any tools for this.
Final Answer: 15

> Finished chain.


# Define Custom Tool

In [ ]:
from langchain.tools import tool

@tool
def word_count(text: str) -> int:
    """Returns the word count."""
    return len(text.split())

In [ ]:
tools

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')),
 StructuredTool(name='openweathermap', description='Get weather for a city', args_schema=<class 'langchain_core.utils.pydantic.openweathermap'>, func=<bound method OpenWeatherMapAPIWrapper.run of OpenWeatherMapAPIWrapper(owm=<pyowm.owm.OWM - API key=************************1f2b0c52, subscription type=free, PyOWM version=('3', '5', '0')>, openweathermap_api_key='b9fc4e6598432c6e3aaadac61f2b0c52')>)]

In [ ]:
tools.append(word_count)
print(tools)

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')), StructuredTool(name='openweathermap', description='Get weather for a city', args_schema=<class 'langchain_core.utils.pydantic.openweathermap'>, func=<bound method OpenWeatherMapAPIWrapper.run of OpenWeatherMapAPIWrapper(owm=<pyowm.owm.OWM - API key=************************1f2b0c52, subscription type=free, PyOWM version=('3', '5', '0')>, openweathermap_api_key='b9fc4e6598432c6e3aaadac61f2b0c52')>), StructuredTool(name='word_count', description='Returns the word count.', args_schema=<class 'langchain_core.utils.pydantic.word_count'>, func=<function word_count at 0x7e7a131c0400>), StructuredTool(name='word_count', description='Returns the word count.', args_schema=<class 'langchain_core.utils.pydantic.word_count'>, func=<function word_count at 0x7e7a131c0400>)]


In [ ]:
# Create a new agent with the DuckDuckGo search tool. OpenWeatherMap tool and custom word count tool
agent = create_react_agent (tools = tools,llm=llm,prompt= prompt)

# Initialize the agent executor with the new agent and tools
agent_executor = AgentExecutor (agent = agent , tools=tools, verbose= True , handle_prasing_errors = True)

In [ ]:
response = agent_executor.invoke({"input": "What is Machine Learning ?"})



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: What is Machine Learning?Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can "learn" the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions. Machine Learning is a technique that allows computers to learn from data and make decisions without explicit programming. It works by identifying patterns in data and using them to make predictions. Machine learning is a subset of AI concerned with training models to allow computers to mimic human thought and decision making without explicit programming. The most common types of ML are supervised learning (learning via labeled data), unsupervised learning (learning via unlabeled data), and reinforcement learning (learning via a reward and punishment respon

In [ ]:
response = agent_executor.invoke({"input": "How many letters in the word  Machine Learning"})



> Entering new AgentExecutor chain...
Thought: The user is asking to count the number of letters in the phrase "Machine Learning". The `word_count` tool counts words, not letters. I need to count the letters manually.
"Machine" has 7 letters.
"Learning" has 8 letters.
Total letters = 7 + 8 = 15.
Final Answer: 15

> Finished chain.


In [ ]:
response = agent_executor.invoke({"input": "What is the temperature of the city where the Lotus tower is loacted"})



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: location of Lotus tower5 days ago - It ranks as the 11th tallest tower in Asia and the 19th tallest tower in the world. Initially, the tower was proposed to be built in Peliyagoda, but the Government of Sri Lanka later decided to change the location. The lotus-shaped structure serves various purposes, including communication ... May 23, 2026 - The tower is on a ridge one kilometre to the southwest of Mount Sir James MacBrien, and though the tower is not prominent in relation to surrounding peaks, it is noted for its sheer rock walls which are home to classic multi-pitch climbing routes. The first ascent was made by William J. Buckingham and party on July 16, 1960, via a traverse from "Tathagata Tower" along the ridge which connects Lotus ... 2 weeks ago - The majority of high-rise structures in the country are located in the commercial capital Colombo. As of May 2017, the tallest structure on the island is 